# Loading files and helper functions

In [1]:
import numpy as np
import pickle
import torch
import numpy as np
from torch.utils.data import Dataset


# TODO: add the right paths
cleaned_adherence_data = np.load("/Users/andreilixandru/Downloads/NDEs-for-adherence-main/reinforce_preprocessed_withDemographics.npy")


def create_feature_mean_dict(train_array):
    # Flatten the first two dimensions (people and sessions) 
    # so we can treat every observation as an independent sample
    # New shape: (n_people * n_sessions, n_dimensions)
    flat_data = train_array.reshape(-1, train_array.shape[-1])
    
    # Extract the relevant dimensions
    # dim_0 = values to average, dim_1 = key 'a', dim_2 = key 'b'
    vals = flat_data[:, 0]
    dim_1 = flat_data[:, 1]
    dim_2 = flat_data[:, 2]
    dim_3 = flat_data[:, 3]
    dim_4 = flat_data[:, 4]
    dim_5 = flat_data[:, 5]
    dim_6 = flat_data[:, 6]
    dim_7 = flat_data[:, 7]
    
    # Combine dim_1, ..., dim_7 into a single array of pairs
    # This makes it easier to find unique combinations
    keys = np.column_stack((dim_1, dim_2, dim_3, dim_4, dim_5, dim_6, dim_7))
    
    # Find unique rows (pairs) and their indices
    unique_keys, indices = np.unique(keys, axis=0, return_inverse=True)
    
    # Calculate means for each unique pair
    # We use np.bincount for speed; it sums 'vals' grouped by 'indices'
    sums = np.bincount(indices, weights=vals)
    counts = np.bincount(indices)
    means = sums / counts
    
    # Zip the unique pairs with their means into the final dictionary
    # Convert keys to tuples so they are hashable
    result_dict = {tuple(key): mean for key, mean in zip(unique_keys, means)}
    
    return result_dict

class AdherenceDataset(Dataset):
    """
    Dataset for full trajectories.
    
    Returns (X, Y) where:
    - X: (T, D-1) control variables over time
    - Y: (T,) adherence target over time (as class indices)
    """
    
    def __init__(self, data, target_as_classes=True):
        """
        Parameters
        ----------
        data : np.ndarray
            Shape (N, T, D) where:
            - data[..., 0] is adherence (can be class indices or continuous values)
            - data[..., 1:] are control variables
        target_as_classes : bool
            If True, treats target as class indices (integers).
            If False, treats target as continuous values (floats).
            Default: True
        """
        self.X = torch.tensor(data[..., 1:], dtype=torch.float32)  # (N, T, D-1)
        
        # Convert target to appropriate type
        if target_as_classes:
            # Ensure target is integer (class indices)
            self.Y = torch.tensor(data[..., 0], dtype=torch.long)   # (N, T)
        else:
            # Keep target as float (continuous values)
            self.Y = torch.tensor(data[..., 0], dtype=torch.float32)   # (N, T)
        
    def __len__(self):
        return self.X.shape[0]
    
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]



# LLM testing pipeline
ends with saving the LLM MSEs in `llm_mses_acrossRuns.pkl`

In [5]:
import os
import numpy as np

# TODO: optional, pick a subset of the data if full dataset makes prompts too large
cleaned_adherence_data = cleaned_adherence_data[:,:,:]

np.set_printoptions(threshold=np.inf)

import numpy as np
from torch.utils.data import random_split
import torch
from sklearn.metrics import mean_squared_error
import numpy as np
import copy

torch.manual_seed(0) # torch seed for reproducibility

# TODO: pick runs
N_RUNS = 3
seeds = [torch.randint(0, 1000000, size=(1,)).item() for _ in range(N_RUNS)]
with open("/Users/andreilixandru/Downloads/NDEs-for-adherence-main/RUNS_reinforce/seeds.pkl", "wb") as f:
    pickle.dump(seeds, f)
TRAIN_SPLIT = 0.8
BATCH_SIZE_VALUE = 4

reinforce_data = cleaned_adherence_data
dataset = AdherenceDataset(reinforce_data, target_as_classes=True)
# dataset = cleaned_adherence_data

dataset_size = len(dataset)
train_size = int(TRAIN_SPLIT * dataset_size)
val_size = dataset_size - train_size


train_losses_acrossRuns= []
llm_mses_acrossRuns = []

val_sets_acrossRuns = []

N = reinforce_data.shape[0]
T = reinforce_data.shape[1]
D = reinforce_data.shape[2]

for run_idx in range(N_RUNS):
    print(f"========= Run {run_idx+1} ============")
    c = 0
    train_set, val_set = random_split(
        dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(seeds[run_idx])
        )

    # ------------------ LLM setup ------------------
    import pickle
    from tarfile import HeaderError
    import numpy as np


    # ------------------ LLM inference ------------------

    num_sessions = cleaned_adherence_data.shape[1]

    y_true_llm = []
    y_pred_llm = []

    # Create val_array of shape (n_persons, n_timepoints, n_dimensions) where dim 0 = targets
    to_insert_in = copy.deepcopy(val_set[:][0])
    to_insert = copy.deepcopy(val_set[:][1])
    val_array = np.insert(to_insert_in, 0, to_insert, axis=2)

    to_insert_in = copy.deepcopy(train_set[:][0])
    to_insert = copy.deepcopy(train_set[:][1])
    train_array = np.insert(to_insert_in, 0, to_insert, axis=2)

    for idx_person in range(len(val_array)):
        for idx_session in range(1, num_sessions): # skipping inferencefor day 1 to match NDE evaluation

            y_pred_llm_oneSession = []
            # Prompt
            past_sessions_data = []
            for s_idx in range(idx_session):
                # Construct the string for each individual past session
                # Customize the indexing below based on your actual data structure
                effort = int(val_array[idx_person, s_idx, 0]) 
                
                session_info = (f"""
                    Day {s_idx + 1}:
                        The fraction of medications you took: {effort}
                        {"Your message reminder invoked the positive outcomes of medication use" if val_array[idx_person, s_idx, 6].item() == 1  else ""}
                        {"Your message reminder invoked the negative outcomes of medication non-use" if val_array[idx_person, s_idx, 7].item() == 1  else ""}
                        {"Your message reminder informed you of your medication adherence in the last week " if val_array[idx_person, s_idx, 2].item() == 1  else ""}
                        {"Your message reminder had a social reinforcement " if val_array[idx_person, s_idx, 3].item() == 1  else ""}
                        {"Your message reminder had a reflective question " if val_array[idx_person, s_idx, 5].item() == 1  else ""}
                    """)
                past_sessions_data.append(session_info)
            # 2. Join them together or leave empty if no sessions exist
            if past_sessions_data:
                past_behavior_header = "\nPast Behavior:\nThe following is your previous medication adherence and the messages you received in your previous days in the program.\n\n"
                past_behavior_text = past_behavior_header + "\n\n".join(past_sessions_data)
            else:
                past_behavior_text = ""

            # Data for the other users
            past_sessions_data2 = []
            for person_idx in range(len(train_array)):
                # Construct the string for each individual past session
                # Customize the indexing below based on your actual data structure
                
                # efforts = train_array[person_idx, :, 0].tolist()
                efforts_mean = [float(train_array[person_idx, :, 0].mean())]
                demographics = train_array[person_idx, 0, 4:].tolist()
                combined = efforts_mean + demographics
                session_info = (
                    " ".join(str(x) for x in combined)
                        )
                past_sessions_data2.append(session_info)

            dict_mean_adherence = create_feature_mean_dict(train_array)
            intervention_combination = (float(train_array[idx_person,idx_session,1]), float(train_array[idx_person,idx_session,2]))

            if intervention_combination in dict_mean_adherence:
                header_activity_users_in_train_set = f"""
                    In case it is useful to make your decision, this is the mean fraction of prescribed medications taken by the other participants in the study that received the same message reminder as you did {dict_mean_adherence[intervention_combination]} and below is an array with the decisions made by other users given their demographics; The rows represent the users, the first value in the row is their mean fraction of prescribed medications they took throughout the study and the rest of the values are these demographics: age_years, number_meds, sex_encoded (0 female, 1 male, edu_level_encoded (0 "Some high school/Below high school", 1 "Some college", 2 "College Grad/Postgrad"), marital_status_encoded (0 Widow/divorced/single/other, 1 Married/partner), race_Asian, race_Black, race_Hispanic, race_Prefer not to say, race_White.
                    """
            else:
                header_activity_users_in_train_set = f"""
                In case it is useful to make your decision, below is an array with the decisions made by other users given their demographics; The rows represent the users, the first value in the row is their mean fraction of prescribed medications they took throughout the study and the rest of the values are these demographics: age_years, number_meds, sex_encoded (0 female, 1 male, edu_level_encoded (0 "Some high school/Below high school", 1 "Some college", 2 "College Grad/Postgrad"), marital_status_encoded (0 Widow/divorced/single/other, 1 Married/partner), race_Asian, race_Black, race_Hispanic, race_Prefer not to say, race_White.
                """

            activity_users_in_train_set = header_activity_users_in_train_set + "\n\n".join(past_sessions_data2) 

            education_levels = ["Some high school/Below high school", "Some college", "College Grad/Postgrad"]
            marital_statuses = ["Widow/divorced/single/other", "Married/partner"]
            
            input_text = f"""
            You are a patient with diabetes enrolled in a study where at each session you receive a reminder to help with your medication adherence.
            Through this program you receive at each day a reminder to take your medication.
            Below is your background and history with the program.

            Your Background:
            You are {val_array[idx_person, idx_session, 8].item()} years old.
            You are prescribed to take {val_array[idx_person, idx_session, 9].item()} medications per day.
            You are {"female" if val_array[idx_person, idx_session, 10].item() == 0  else "male"}.
            Your education level is {education_levels[int(val_array[idx_person, idx_session, 11].item())]}.
            Your marital status is {marital_statuses[int(val_array[idx_person, idx_session, 12].item())]}.
            {"Your race is Asian." if val_array[idx_person, idx_session, 13].item() == 1  else ""}
            {"Your race is Black/African American." if val_array[idx_person, idx_session, 14].item() == 1  else ""}
            {"Your race is Hispanic/Latinx." if val_array[idx_person, idx_session, 15].item() == 1  else ""}
            {"" if val_array[idx_person, idx_session, 16].item() == 1 else ""}
            {"Your race is White." if val_array[idx_person, idx_session, 17].item() == 1  else ""}

            {past_behavior_text}

            Based on this information, the context of the program the typical behavior of patients with diabetes, and the fact that in the session today you got a reminder to help you take your medication, decide what your adherence level will be by picking a real number in the interval between 0 and 1, representing the fraction of your prescribed medications that you took today (that is, 0 meaning you didn't take any of your medications, 1 meaning you took all your medication, 0.5 meaning you took half of your medication).
            Key Consideration: The adherence from a previous day does not necessarily imply the adherence at the next. The effort should depend on your specific circumstances on the day of that session.


            {"Today's message reminder invoked the positive outcomes of medication use" if val_array[idx_person, s_idx, 6].item() == 1  else ""}
            {"Today's message reminder invoked the negative outcomes of medication non-use" if val_array[idx_person, s_idx, 7].item() == 1  else ""}
            {"Today's message reminder informed you of your medication adherence in the last week " if val_array[idx_person, s_idx, 2].item() == 1  else ""}
            {"Today's message reminder had a social reinforcement " if val_array[idx_person, s_idx, 3].item() == 1  else ""}
            {"Today's message reminder had a reflective question " if val_array[idx_person, s_idx, 5].item() == 1  else ""}

            Question: Today you got a message reminder to help you take your medication, decide what your adherence level will be by picking a real number in the interval between 0 and 1, representing the fraction of your prescribed medications that you took today.
            Please respond with your final decision by only picking a real number in the interval between 0 and 1, no other text should be included.
            {activity_users_in_train_set}
            """

            # ------------------ LLM evaluation ------------------
            import anthropic
            # TODO: Add your API key
            client = anthropic.Anthropic(api_key='sk-ant-XXX')

            message = client.messages.create(
                model="claude-haiku-4-5",
                max_tokens=1000,
                messages=[
                    {
                        "role": "user",
                        "content": input_text,
                    }
                ],
            )
            content = message.content[0].text
            if c % 10 == 0:
                print(f"Inference number {c}")
            c = c+1
            # ------------------ End section to change ------------------
            print("true:", int(val_array[idx_person, idx_session, 0]))
            try:
                y_pred_llm.append(int(content)) 
                print("pred: ", content)
            except (ValueError, TypeError):         # in case the llm outputs also its thinking, we take the last integer in the output
                import re
                numbers = re.findall(r'\d+(?:\.\d+)?', content)
                y_pred_llm.append(numbers[-1])
                print("cleaned pred:", numbers[-1])
            
            y_true_llm.append(val_array[idx_person, idx_session, 0])

    with open(f"/Users/andreilixandru/Downloads/NDEs-for-adherence-main/RUNS_reinforce/pred_run_{run_idx+1}.pkl", "wb") as f:
      pickle.dump(y_pred_llm, f) 
    with open(f"/Users/andreilixandru/Downloads/NDEs-for-adherence-main/RUNS_reinforce/true_run_{run_idx+1}.pkl", "wb") as f:
      pickle.dump(y_true_llm, f)
    mse_llm_run = mean_squared_error(y_true_llm, y_pred_llm)
    llm_mses_acrossRuns.append(mse_llm_run)
    with open(f"/Users/andreilixandru/Downloads/NDEs-for-adherence-main/RUNS_reinforce/mse_run_{run_idx+1}.pkl", "wb") as f:
      pickle.dump(mse_llm_run, f) 


# Save LLM MSEs across runs
with open("/Users/andreilixandru/Downloads/NDEs-for-adherence-main/RUNS_reinforce/llm_mses_acrossRuns.pkl", "wb") as f:
    pickle.dump(llm_mses_acrossRuns, f)


========= Run 1 ============
Inference number 0
true: 0
cleaned pred: 0.05
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.15
true: 0
cleaned pred: 0.05
true: 0
cleaned pred: 0.05
true: 0
cleaned pred: 0.02
true: 0
cleaned pred: 0.05
true: 0
cleaned pred: 0.0
Inference number 10
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
Inference number 20
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
Inference number 30
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
true: 0
cleaned pred: 0.0
tr

In [6]:
val_array[30, 0, 5].item()

4.0

In [20]:
bin_number = int(val_array[18, 0, 13])
age_intervals = ["18-20", "21-25", "26-30", "31-35", "36-40", "41-45", "46-50", "51-55", "56-60", "61-65", "66-70", "71-74"]
if 0 <= bin_number < len(age_intervals):
    # Retrieve the string directly
    interval = age_intervals[bin_number]
    age_message = f"Your age is between {interval} years old."
print(bin_number)
print(age_message)

3
Your age is between 31-35 years old.


In [7]:
y_pred_llm_oneSession = []
# Prompt
past_sessions_data = []
for s_idx in range(idx_session):
    # Construct the string for each individual past session
    # Customize the indexing below based on your actual data structure
    effort = int(val_array[idx_person, s_idx, 0]) 
    
    session_info = (f"""
        Day {s_idx + 1}:
            The fraction of medications you took: {effort}
            {"Your message reminder invoked the positive outcomes of medication use" if val_array[idx_person, s_idx, 6].item() == 1  else ""}
            {"Your message reminder invoked the negative outcomes of medication non-use" if val_array[idx_person, s_idx, 7].item() == 1  else ""}
            {"Your message reminder informed you of your medication adherence in the last week " if val_array[idx_person, s_idx, 2].item() == 1  else ""}
            {"Your message reminder had a social reinforcement " if val_array[idx_person, s_idx, 3].item() == 1  else ""}
            {"Your message reminder had a reflective question " if val_array[idx_person, s_idx, 5].item() == 1  else ""}
        """)
    past_sessions_data.append(session_info)
# 2. Join them together or leave empty if no sessions exist
if past_sessions_data:
    past_behavior_header = "\nPast Behavior:\nThe following is your previous medication adherence and the messages you received in your previous days in the program.\n\n"
    past_behavior_text = past_behavior_header + "\n\n".join(past_sessions_data)
else:
    past_behavior_text = ""

# Data for the other users
past_sessions_data2 = []
for person_idx in range(len(train_array)):
    # Construct the string for each individual past session
    # Customize the indexing below based on your actual data structure
    
    # efforts = train_array[person_idx, :, 0].tolist()
    efforts_mean = [float(train_array[person_idx, :, 0].mean())]
    demographics = train_array[person_idx, 0, 4:].tolist()
    combined = efforts_mean + demographics
    session_info = (
        " ".join(str(x) for x in combined)
            )
    past_sessions_data2.append(session_info)

dict_mean_adherence = create_feature_mean_dict(train_array)
intervention_combination = (float(train_array[idx_person,idx_session,1]), float(train_array[idx_person,idx_session,2]))

if intervention_combination in dict_mean_adherence:
    header_activity_users_in_train_set = f"""
        In case it is useful to make your decision, this is the mean fraction of prescribed medications taken by the other participants in the study that received the same message reminder as you did {dict_mean_adherence[intervention_combination]} and below is an array with the decisions made by other users given their demographics; The rows represent the users, the first value in the row is their mean fraction of prescribed medications they took throughout the study and the rest of the values are these demographics: age_years, number_meds, sex_encoded (0 female, 1 male, edu_level_encoded (0 "Some high school/Below high school", 1 "Some college", 2 "College Grad/Postgrad"), marital_status_encoded (0 Widow/divorced/single/other, 1 Married/partner), race_Asian, race_Black, race_Hispanic, race_Prefer not to say, race_White.
        """
else:
    header_activity_users_in_train_set = f"""
    In case it is useful to make your decision, below is an array with the decisions made by other users given their demographics; The rows represent the users, the first value in the row is their mean fraction of prescribed medications they took throughout the study and the rest of the values are these demographics: age_years, number_meds, sex_encoded (0 female, 1 male, edu_level_encoded (0 "Some high school/Below high school", 1 "Some college", 2 "College Grad/Postgrad"), marital_status_encoded (0 Widow/divorced/single/other, 1 Married/partner), race_Asian, race_Black, race_Hispanic, race_Prefer not to say, race_White.
    """

activity_users_in_train_set = header_activity_users_in_train_set + "\n\n".join(past_sessions_data2) 

education_levels = ["Some high school/Below high school", "Some college", "College Grad/Postgrad"]
marital_statuses = ["Widow/divorced/single/other", "Married/partner"]


print( "unique values in education:", val_array[:, :, 11].unique())

input_text = f"""
You are a patient with diabetes enrolled in a study where at each session you receive a reminder to help with your medication adherence.
Through this program you receive at each day a reminder to take your medication.
Below is your background and history with the program.

Your Background:
You are {val_array[idx_person, idx_session, 8].item()} years old.
You are prescribed to take {val_array[idx_person, idx_session, 9].item()} medications per day.
You are {"female" if val_array[idx_person, idx_session, 10].item() == 0  else "male"}
Your education level is {education_levels[int(val_array[idx_person, idx_session, 11].item())]}
Your marital status is {marital_statuses[int(val_array[idx_person, idx_session, 12].item())]}
{"Your race is Asian" if val_array[idx_person, idx_session, 13].item() == 1  else ""}
{"Your race is Black/African American" if val_array[idx_person, idx_session, 14].item() == 1  else ""}
{"Your race is Hispanic/Latinx" if val_array[idx_person, idx_session, 15].item() == 1  else ""}
{"" if val_array[idx_person, idx_session, 16].item() == 1 else ""}
{"Your race is White" if val_array[idx_person, idx_session, 17].item() == 1  else ""}

{past_behavior_text}

Based on this information, the context of the program the typical behavior of patients with diabetes, and the fact that in the session today you got a reminder to help you take your medication, decide what your adherence level will be by picking a real number in the interval between 0 and 1, representing the fraction of your prescribed medications that you took today (that is, 0 meaning you didn't take any of your medications, 1 meaning you took all your medication, 0.5 meaning you took half of your medication).
Key Consideration: The adherence from a previous day does not necessarily imply the adherence at the next. The effort should depend on your specific circumstances on the day of that session.


{"Today's message reminder invoked the positive outcomes of medication use" if val_array[idx_person, s_idx, 6].item() == 1  else ""}
{"Today's message reminder invoked the negative outcomes of medication non-use" if val_array[idx_person, s_idx, 7].item() == 1  else ""}
{"Today's message reminder informed you of your medication adherence in the last week " if val_array[idx_person, s_idx, 2].item() == 1  else ""}
{"Today's message reminder had a social reinforcement " if val_array[idx_person, s_idx, 3].item() == 1  else ""}
{"Today's message reminder had a reflective question " if val_array[idx_person, s_idx, 5].item() == 1  else ""}

Question: Today you got a message reminder to help you take your medication, decide what your adherence level will be by picking a real number in the interval between 0 and 1, representing the fraction of your prescribed medications that you took today.
Please respond with your final decision by only picking a real number in the interval between 0 and 1, no other text should be included.
{activity_users_in_train_set}
"""

print(input_text)

unique values in education: tensor([1., 2.])

You are a patient with diabetes enrolled in a study where at each session you receive a reminder to help with your medication adherence.
Through this program you receive at each day a reminder to take your medication.
Below is your background and history with the program.

Your Background:
You are 73.0 years old.
You are prescribed to take 2.0 medications per day.
You are male
Your education level is Some college
Your marital status is Widow/divorced/single/other

Your race is Black/African American





Past Behavior:
The following is your previous medication adherence and the messages you received in your previous days in the program.


        Day 1:
            The fraction of medications you took: 0
            
            Your message reminder invoked the negative outcomes of medication non-use
            
            Your message reminder had a social reinforcement 
            Your message reminder had a reflective question 
     